# Baseline на задачах MiMo (Colab A100 40 ГБ)

Модель крутится здесь (vLLM), инструменты и проверка тестов идут на ноутбуке с Docker через туннель
(`sim/mimo_server.py`). Перед запуском на ноутбуке должны работать: сервер песочницы и
`cloudflared tunnel`. Токены вводятся в ячейках, в репозиторий не попадают.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 1. Код и зависимости

In [ ]:
import os
if not os.path.exists("gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git
%cd gemma4-swe-rl
!git pull -q
# torch не ставим: его приносит vllm. Нужны только клиентские пакеты.
!pip install -q vllm openai pyyaml requests huggingface_hub kagglehub
!python -c "import vllm; print('vllm', vllm.__version__)"

## 2. Связь с ноутбуком (адрес туннеля и токен вводятся вручную)

In [ ]:
import getpass, os, json, urllib.request
os.environ["SWE_SANDBOX_URL"] = input("Адрес туннеля (https://....trycloudflare.com): ").strip().rstrip("/")
os.environ["SWE_SANDBOX_TOKEN"] = getpass.getpass("Токен сервера песочницы: ").strip()

print(urllib.request.urlopen(os.environ["SWE_SANDBOX_URL"] + "/health", timeout=30).read())
req = urllib.request.Request(os.environ["SWE_SANDBOX_URL"] + "/tasks",
                             headers={"Authorization": "Bearer " + os.environ["SWE_SANDBOX_TOKEN"]})
ids = json.load(urllib.request.urlopen(req, timeout=30))["result"]
print(len(ids), "задач, первые:", ids[:3])

## 3. Веса модели

Откуда брать веса `gemma-4-31b-it-qat-w4a16-ct`, в регламенте не указано, поэтому ниже два
варианта, оба с ПРЕДПОЛОЖИТЕЛЬНЫМ именем репозитория (не проверено). Если оба упадут, пришлите
текст ошибки. Для Hugging Face может понадобиться токен и принятие лицензии на странице модели.

In [ ]:
import getpass, os
MODEL_DIR = "/content/model"
HF_REPO = "google/gemma-4-31b-it-qat-w4a16-ct"            # предположение
KAGGLE_MODEL = "google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct"   # предположение

if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download(HF_REPO, local_dir=MODEL_DIR, token=hf_token)
        print("скачано с Hugging Face")
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download(KAGGLE_MODEL)
        !cp -r "{path}"/. {MODEL_DIR}/
        print("скачано с Kaggle Models")
!ls {MODEL_DIR} | head -20; du -sh {MODEL_DIR}

## 4. Сервер vLLM

Параметры как в боевом запуске: `max_model_len=32768`, парсеры `gemma4`, `enable_thinking=True`.
Отличие: одна карта, поэтому без tensor-parallel и с `gpu-memory-utilization 0.92`. Если сервер
упадёт с ошибкой про нехватку памяти под KV-кэш, уменьшите `--max-model-len` до 24576 (результаты
тогда не строго сопоставимы с боевыми).

In [ ]:
import subprocess, time, requests, os
MODEL_DIR = "/content/model"
cmd = ["vllm", "serve", MODEL_DIR, "--served-model-name", "gemma-4-31b-it-qat-w4a16-ct",
       "--quantization", "compressed-tensors", "--dtype", "bfloat16",
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.92", "--max-num-seqs", "4",
       "--enable-auto-tool-choice", "--tool-call-parser", "gemma4", "--reasoning-parser", "gemma4",
       "--default-chat-template-kwargs", '{"enable_thinking": true}',
       "--port", "8000"]
log = open("/content/vllm.log", "w")
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while time.time() - t0 < 1200:
    if proc.poll() is not None:
        break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=3).status_code == 200:
            print("vLLM готов за %.0f c" % (time.time() - t0)); break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print("не дождались")
if proc.poll() is not None:
    print("vLLM упал, хвост лога:"); print(open("/content/vllm.log").read()[-3000:])

## 5. Проверка: один запрос с инструментами

In [ ]:
from llm.llm_client import LLMClient
from sim.tool_schemas import ROOT_TOOL_SCHEMAS
r = LLMClient().chat([{"role": "user", "content": "Run `ls` using run_command."}], ROOT_TOOL_SCHEMAS, temperature=0.2)
print(r["tool_calls"], r["finish_reason"], len(r["token_logprobs"]))

## 6. Baseline

Первый эпизод каждой задачи может ждать, пока ноутбук скачает образ (до десятков минут).
`--workers` не больше `--max-sessions` сервера на ноутбуке (сейчас 2). Прогон можно прервать и
запустить заново: готовые задачи пропускаются.

In [ ]:
!python -m sim.run_mimo_baseline --first 15 --workers 2 --out results/mimo_baseline

## 7. Результаты

In [ ]:
import json
for l in open("results/mimo_baseline/results.jsonl", encoding="utf-8"):
    r = json.loads(l)
    print(r["instance_id"], r["passed"], r.get("result", {}).get("reason"), r.get("tool_calls"), r["seconds"], r["error"])
!zip -qr mimo_baseline.zip results/mimo_baseline && echo "mimo_baseline.zip готов (скачайте из файлов слева)"